# Sharing Assets Between Workspaces using regestries


The tutorial is based on the following materials:
- [Machine Learning registries for MLOps](https://learn.microsoft.com/en-us/azure/machine-learning/concept-machine-learning-registries-mlops?view=azureml-api-2)
- [Manage Azure Machine Learning registries](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-manage-registries?view=azureml-api-2&tabs=cli)
- [Share models, components, and environments across workspaces with registries](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-share-models-pipelines-across-workspaces-with-registries?view=azureml-api-2&tabs=python)
- [Share data across workspaces with registries](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-share-data-across-workspaces-with-registries?view=azureml-api-2&tabs=cli)
<!-- - []() -->


# Notebook Setup

Set project paths and load workspace MLClient.

In [1]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
# Set the path to your project root manually if the above code does not work
# p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Starting working directory: c:\Users\dmika\DEV\Projects-local\dp100-learn\development-in-progress
Changed working directory to: c:\Users\dmika\DEV\Projects-local\dp100-learn
Added to sys.path: C:\Users\dmika\DEV\Projects-local\dp100-learn
Added to sys.path: C:\Users\dmika\DEV\Projects-local\dp100-learn


# Create Registry

In [10]:
registry_file_path = "assets/tutorials-materials/configs/registry.yml"
registry_name = "dmdp100-registry"
registry_location = "westeurope"
parent_image = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"

env_content = f"""
name: {registry_name}
tags:
  description: Basic registry with one primary region and to additional regions
  foo: bar
location: {registry_location}
replication_locations:
  - location: {registry_location}
  - location: swedencentral
"""
with open(registry_file_path, "w") as f:
    f.write(env_content)

In [ ]:
!az ml registry create --resource-group azure-ml-dev-rg --name dmdp100-registry --file assets/tutorials-materials/configs/registry.yml

............{
  "containerRegistry": null,
  "description": null,
  "discoveryUrl": "https://westeurope.api.azureml.ms/registrymanagement/v1.0/registries/dmdp100-registry/discovery",
  "identity": {
    "principalId": "2787d331-6ed6-474c-9ef8-0a1df31c7871",
    "tenantId": "50c76291-0c80-4444-a2fb-4f8ab168c311",
    "type": "SystemAssigned",
    "userAssignedIdentities": null
  },
  "intellectualProperty": null,
  "location": "westeurope",
  "managedResourceGroup": {
    "resourceId": "/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/azureml-rg-dmdp100-registry_176eb72d-b721-4444-b2fa-ac9924ed4ace"
  },
  "mlflowRegistryUri": "azureml://westeurope.api.azureml.ms/mlflow/v1.0/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/azure-ml-dev-rg/providers/Microsoft.MachineLearningServices/registries/dmdp100-registry",
  "name": "dmdp100-registry",
  "properties": {},
  "publicNetworkAccess": "Enabled",
  "replicationLocations": [
    {
      "acrConfig": [
  

Class RegistryRegionDetailsSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


# Connect to a Registry

In [11]:
from azure.ai.ml import MLClient

ml_client_registry = MLClient(
    credential=ml_client._credential,
    registry_name=registry_name,
    registry_location=registry_location
)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


# Create Assets in a Registry

## Create an Environment

In [4]:
env_file_path = "assets/tutorials-materials/configs/registry_env.yml"
main_env_name = "registry_env"
parent_image = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"

env_content = f"""
channels:
  - conda-forge
dependencies:
  - python=3.10.11
  - pip=22.3.1
  - pip:
      - scipy
      - numpy
      - pandas
      - scikit-learn
name: {main_env_name}
"""
with open(env_file_path, "w") as f:
    f.write(env_content)

In [12]:
from azure.ai.ml.entities import Environment

env = Environment(
    name=main_env_name,
    image=parent_image,
    conda_file=env_file_path,
    description="Test environment to share between workspaces usign registries."
)
ml_client_registry.environments.create_or_update(env)

Subtype value SAS has no mapping, use base class DataReferenceCredentialDto.


Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'registry_env', 'description': 'Test environment to share between workspaces usign registries.', 'tags': {}, 'properties': {'azureml.labels': 'default,latest,invisibleLatest'}, 'print_as_yaml': False, 'id': 'azureml://registries/dmdp100-registry/environments/registry_env/versions/1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x0000028E543FFA30>, 'serialize': <msrest.serialization.Serializer object at 0x0000028E543DA110>, 'version': '1', 'conda_file': {'channels': ['conda-forge'], 'dependencies': ['python=3.10.11', 'pip=22.3.1', {'pip': ['scipy', 'numpy', 'pandas', 'scikit-learn']}], 'name': 'registry_env